## Agent Loop using todos to Build Task Solver from Scratch

Based on Latest Emerging Definition of AI Agent: "An LLM agent runs tools in a loop to achieve a goal".

## Sports Bet Analyzer Agent

An autonomous agent that evaluates sports betting opportunities using an agentic loop pattern.

Given a natural language question about a potential bet, the agent creates a structured 
analysis plan and executes each step in sequence using tools, ultimately mimicking how a sharp bettor or quant analyst would break down a wager before placing it (using their probability vs. best odds, incorporating their bankroll for a final recommendation).

### What it analyzes:
- Implied probability from the odds
- Expected value (EV) of the bet
- Recommended bet size using Kelly Criterion bankroll management
- A final recommendation with reasoning

### Agentic Pattern:
The agent loop runs until the task is complete — the LLM decides when to call tools 
and when it has enough information to deliver a final answer. No human in the loop.

### Tools:
- `create_todos` — breaks the problem into an ordered analysis plan
- `mark_complete` — executes and logs each step with notes
- `calculate_ev` — computes expected value given odds and estimated win probability
- `kelly_criterion` — computes recommended bet size given bankroll and edge

### Example question:
"The Bills are favored at -125 against the Chiefs. I think Buffalo wins 58% of the time. 
My bankroll is $1000. Should I bet and how much?"

In [34]:
# imports 

#rich for making formatted text output in the terminal
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [35]:
# gpt model
openai = OpenAI()

In [36]:
# shows text in the terminal using rich ideally
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [37]:
# lists to keep track of things the agent needs to do, has done, and has seen.
todos = []
completed = []

In [38]:
# loops through todos and completed and returns a string showing each todo /whether its done
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [39]:
# adds new todos to the list, marks them all as False in completed, prints the notes, 
# then calls get todo report
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [40]:
# marks a todo item as done and prints the notes, then calls get todo report
def mark_todo_completed(index: int, completion_notes: str = "") -> str:
    # LLM sends 1-based index, convert to 0-based
    zero_index = index - 1
    if 0 <= zero_index < len(todos):
        completed[zero_index] = True
        if completion_notes:
            show(f"Marked Todo #{index} as completed. Notes: {completion_notes}")
        else:
            show(f"Marked Todo #{index} as completed.")
    else:
        show(f"Invalid index: {index}. No such todo.")
    return get_todo_report()

### Domain-Specific Tools:

1. EV Calculator

In [41]:
def calculate_ev(win_probability: float, odds: int) -> str:
    if odds < 0:
        decimal_odds = 1 + (100 / abs(odds))
    else:
        decimal_odds = 1 + (odds / 100)
    ev = (win_probability * (decimal_odds - 1)) - (1 - win_probability)
    show(f"Calculated EV: {ev:.4f}")
    return f"EV: {ev:.4f}"

2. Kelly Criterion

In [42]:
def kelly_criterion(win_probability: float, odds: int, bankroll: float) -> str:
    if odds < 0:
        decimal_odds = 1 + (100 / abs(odds))
    else:
        decimal_odds = 1 + (odds / 100)
    kelly_fraction = ((decimal_odds - 1) * win_probability - (1 - win_probability)) / (decimal_odds - 1)
    if kelly_fraction <= 0:
        return "No edge. Do not bet."

    show(f"Kelly Criterion Fraction: {kelly_fraction:.4f}")

    # recommended bet
    recommended_bet = kelly_fraction * bankroll
    return f"Kelly Fraction: {kelly_fraction:.4f} | Recommended Bet: ${recommended_bet:.2f}"


### JSON tool definitions 

In [43]:
# create todo json
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [44]:
# mark complete json
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [45]:
# ev calculator json
calculate_ev_json = {
    "name": "calculate_ev",
    "description": "Calculate the expected value of a bet given the win probability and odds",
    "parameters": {
        "type": "object",
        "properties": {
            "win_probability": {
                "description": "The probability of winning the bet (between 0 and 1)",
                "type": "number",
                "minimum": 0,
                "maximum": 1
            },
            "odds": {
                "description": "The odds of the bet in American format (e.g. -150 or +200)",
                "type": "integer"
            }
        },
        "required": ["win_probability", "odds"],
        "additionalProperties": False
    }
}  

In [46]:
# kelly criterion json
kelly_criterion_json = {
    "name": "kelly_criterion",
    "description": "Calculate the Kelly Criterion fraction and recommended bet size given the win probability, odds, and bankroll",
    "parameters": {
        "type": "object",
        "properties": {
            "win_probability": {
                "description": "The probability of winning the bet (between 0 and 1)",
                "type": "number",             
                "minimum": 0,               
                "maximum": 1
            },
            "odds": {
                "description": "The odds of the bet in American format (e.g. -150 or +200)",
                "type": "integer"
            },
            "bankroll": {
                "description": "The total bankroll available for betting",
                "type": "number",
                "minimum": 0
            }
        },
        "required": ["win_probability", "odds", "bankroll"],
        "additionalProperties": False
    }
}  

In [47]:
# bundle them into a tools list for agent use
tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json},
    {"type": "function", "function": calculate_ev_json},
    {"type": "function", "function": kelly_criterion_json},
]

### Handle Tool Calls Function

In [48]:
# function actually executing the tools based on tool calls from the agent
# reads tool name and arguments from LLMs response
# finds and calls the python function by its name
# returns result in format to LLM expects so it can continue its reasoning
def handle_tool_calls(tool_calls):
    tool_map = {
        "create_todos": create_todos,
        "mark_complete": mark_todo_completed,
        "calculate_ev": calculate_ev,
        "kelly_criterion": kelly_criterion
    }
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        if tool_name in tool_map:
            try:
                result = tool_map[tool_name](**arguments)
            except Exception as e:
                result = f"Error executing tool '{tool_name}': {str(e)}"
                show(result)
        else:
            result = f"Tool '{tool_name}' not found."
            show(result)
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

### Loop Function

In [49]:
# call LLM in while loop, check finish reason on the response, if 
# tool_calls present, handle them, append results to messages, and loop again
# break otherwise, and show final response in rich console
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

### System Message to LLM

In [56]:
system_message = """
You are a sharp sports betting analyst. When given a betting question, you use your 
todo tools to create a structured analysis plan, then execute each step in sequence.

Your analysis must always:
1. Create a todo list breaking down the analysis steps
2. Calculate the expected value (EV) of the bet using calculate_ev
3. Calculate the recommended bet size using kelly_criterion
4. Mark each step complete with detailed notes as you go
5. Finish with a clear final recommendation — bet or no bet, and why

Be concise and sharp. Think like a quant, not a gambler. Create your todo list plan first,
then execute each step one at a time. Do not mark todos complete until you have actually
run the tool for that step.
Do not ask clarifying questions — work with what you're given and deliver a verdict.
Provide your final answer in Rich console markup without code blocks.
"""

test_user_message = """
The Buffalo Bills are favored at -110 against the Kansas City Chiefs.
I've done my own analysis and believe Buffalo has a 58% chance of winning.
My current bankroll is $1000.

Should I place this bet? If so, how much should I wager?
"""

### Run Loop

In [57]:
todos, completed = [], []

# system = background instructions above
# user = humans message (betting question)
# assistance = LLMs response (getting appended as convo grows)
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": test_user_message}]

In [58]:
# call
#loop(messages)

In [59]:
print("Enter your bet details (odds, your win probability, bankroll):")
user_message = input("Enter your betting question: ")
messages_input = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

print(user_message)
loop(messages_input)

Enter your bet details (odds, your win probability, bankroll):
I want to bet on the Toronto Raptors ML at odds +360. I have them at 22% probability to win with my model. My bankroll is $1,000. 


Todo #1: Calculate the expected value of the bet using the provided probability and odds.
Todo #2: Calculate the recommended bet size using the Kelly Criterion based on the probability, odds, and bankroll.

Calculated EV: 0.0120

Kelly Criterion Fraction: 0.0033

Marked Todo #1 as completed. Notes: Calculated the expected value (EV) of the bet: EV = 0.0120, indicating a small 
positive expected profit.

Todo #1: Calculate the expected value of the bet using the provided probability and odds.
Todo #2: Calculate the recommended bet size using the Kelly Criterion based on the probability, odds, and bankroll.

Marked Todo #2 as completed. Notes: Calculated the recommended bet size using the Kelly Criterion: Kelly Fraction =
0.0033, which suggests betting $3.33 from a $1,000 bankroll.

Todo #1: Calculate the expected value of the bet using the provided probability and odds.
Todo #2: Calculate the recommended bet size using the Kelly Criterion based on the probability, odds, and bankroll.

**Final Recommendation**: Bet **$3.33** on the Toronto Raptors ML at odds +360.

- **Expected Value (EV)**: 0.0120 indicates a small positive expected profit, meaning that over time, this bet 
could be profitable.
- **Kelly Criterion Recommended Bet**: The calculated Kelly Fraction suggests betting $3.33 from a bankroll of 
$1,000.

This represents a favorable situation based on your model's probability of winning, which suggests a positive 
expected value. Thus, placing the bet aligns with a calculated approach to betting management.